# Stage 2 — Subtask 3: Taxonomy Harmonisation
**Leaf-JEPA IRP** | Dataset Preparation

This notebook:
1. Loads the taxonomy from ST1
2. Builds the PlantVillage ↔ PlantDoc cross-dataset mapping
3. Defines three evaluation label spaces: full PV, cross-domain, zero-shot
4. Validates Healthy class inclusion
5. Exports `taxonomy_artefacts.json` — consumed by evaluation scripts


In [31]:
import json, sys
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve().parent
OUTPUTS_DIR  = PROJECT_ROOT / "stage2_dataset_preparation/outputs"
TAXONOMY_OUT = OUTPUTS_DIR / "taxonomy"
TAXONOMY_OUT.mkdir(parents=True, exist_ok=True)


# ── local dataset locations ───────────────────────
PLANTVILLAGE_RAW_DIR = Path("../../data/plantvillage_raw")
PLANTDOC_RAW_DIR     = Path("../../data/plantdoc_raw")
VALID_EXT   = {".jpg", ".jpeg", ".png"}


taxonomy_data = json.loads((OUTPUTS_DIR / "taxonomy.json").read_text())
entries = taxonomy_data["entries"]
print(f"Loaded {len(entries)} taxonomy entries")


Loaded 37 taxonomy entries


## 1. PlantDoc Full Class List

Source: Singh et al. (2020). These are the exact folder names in the published dataset.


In [41]:
if not PLANTDOC_RAW_DIR.exists():
    print(f"⚠️  Not found: {PLANTDOC_RAW_DIR}  — update paths above to match your filesystem")
class_map = {}
pd_class_names = []
for cls_dir in sorted(PLANTDOC_RAW_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    imgs = sorted([p for p in cls_dir.iterdir() if p.suffix.lower() in VALID_EXT])
    if imgs:
        class_name = cls_dir.name
        class_map[cls_dir.name] = imgs
        pd_class_names.append(class_name)
print(f"  {PLANTDOC_RAW_DIR.name}: {len(class_map)} classes, {sum(len(v) for v in class_map.values()):,} images")



  plantdoc_raw: 28 classes, 2,578 images


## 2. Healthy Class Validation

> **Common pitfall:** Accidentally excluding the Healthy class produces a model incapable of negative predictions. This assertion must pass before any further processing.


In [13]:
healthy = [e for e in entries if e["pathogen_category"] == "Healthy"]
assert len(healthy) > 0, (
    "CRITICAL: No Healthy entries found. "
    "A model without a Healthy class cannot distinguish healthy from diseased tissue."
)
print(f"✅ Healthy class: {len(healthy)} entries across {len({e['crop'] for e in healthy})} crops")
for h in healthy:
    assert "healthy" in h["plantvillage_label"].lower(), f"Possible misclassification: {h['plantvillage_label']}"
print("✅ All Healthy entries have 'healthy' in PlantVillage label — no misclassification")


✅ Healthy class: 11 entries across 11 crops
✅ All Healthy entries have 'healthy' in PlantVillage label — no misclassification


## 3. Cross-Dataset Mapping

Three categories:
- **SHARED** — disease-crop combination in both datasets → cross-domain evaluation
- **PV ONLY** — PlantVillage exclusive → trained on, not evaluatable on PlantDoc
- **PlantDoc ONLY** — zero-shot test (model never saw these diseases during training)


In [34]:
shared, pv_only = [], []
mapped_pd = set()

for entry in entries:
    if entry.get("plantdoc_label"):
        shared.append({
            "plantvillage_label": entry["plantvillage_label"],
            "plantdoc_label":     entry["plantdoc_label"],
            "canonical_name":     entry["common_name"],
            "eppo_pathogen":      entry.get("eppo_pathogen"),
            "pathogen_category":  entry["pathogen_category"],
        })
        mapped_pd.add(entry["plantdoc_label"])
    else:
        pv_only.append({"plantvillage_label": entry["plantvillage_label"],
                        "common_name": entry["common_name"]})

pd_only = [{"plantdoc_label": c, "note": "Zero-shot: no PlantVillage equivalent"}
           for c in class_map if c not in mapped_pd]

print(f"Shared (PV ∩ PlantDoc): {len(shared):>3} classes  — cross-domain evaluation set")
print(f"PlantVillage only     : {len(pv_only):>3} classes  — trained, not transferable")
print(f"PlantDoc only         : {len(pd_only):>3} classes  — zero-shot test")
print(f"\n{'='*55}")
print(f"Cross-domain evaluation uses {len(shared)} shared classes.")
print(f"{len(pd_only)} PlantDoc-only classes form a zero-shot sub-experiment.")

# Show shared mapping
pd.DataFrame(shared)[["canonical_name","plantvillage_label","plantdoc_label","pathogen_category"]].head(10)


Shared (PV ∩ PlantDoc):  16 classes  — cross-domain evaluation set
PlantVillage only     :  21 classes  — trained, not transferable
PlantDoc only         :  12 classes  — zero-shot test

Cross-domain evaluation uses 16 shared classes.
12 PlantDoc-only classes form a zero-shot sub-experiment.


,canonical_name,plantvillage_label,plantdoc_label,pathogen_category
0,Apple Scab,Apple___Apple_scab,Apple Scab Leaf,Fungal
1,Gray Leaf Spot (Corn),Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,Corn Gray leaf spot,Fungal
2,Common Rust (Corn),Corn_(maize)___Common_rust_,Corn rust leaf,Fungal
3,Northern Leaf Blight (Corn),Corn_(maize)___Northern_Leaf_Blight,Corn leaf blight,Fungal
4,Grape Black Rot,Grape___Black_rot,grape leaf black rot,Fungal
5,Potato Early Blight,Potato___Early_blight,Potato leaf early blight,Fungal
6,Potato Late Blight,Potato___Late_blight,Potato leaf late blight,Fungal
7,Tomato Early Blight,Tomato___Early_blight,Tomato Early blight leaf,Fungal
8,Tomato Late Blight,Tomato___Late_blight,Tomato leaf late blight,Fungal
9,Tomato Septoria Leaf Spot,Tomato___Septoria_leaf_spot,Tomato Septoria leaf spot,Fungal


## 4. Evaluation Label Spaces

In [35]:
pv_all_labels  = sorted({e["plantvillage_label"] for e in entries})
shared_pv_lbls = sorted([e["plantvillage_label"] for e in shared])
shared_pd_lbls = sorted([e["plantdoc_label"]     for e in shared])
zero_shot_lbls = sorted([e["plantdoc_label"]     for e in pd_only])

eval_spaces = {
    "plantvillage_full": {
        "classes": pv_all_labels, "num_classes": len(pv_all_labels),
        "label_to_idx": {c: i for i, c in enumerate(pv_all_labels)},
        "usage": "All PlantVillage experiments",
    },
    "cross_domain_pv_labels": {
        "classes": shared_pv_lbls, "num_classes": len(shared_pv_lbls),
        "label_to_idx": {c: i for i, c in enumerate(shared_pv_lbls)},
        "usage": "Shared PlantVillage labels for cross-domain evaluation (RQ4)",
    },
    "cross_domain_pd_labels": {
        "classes": shared_pd_lbls, "num_classes": len(shared_pd_lbls),
        "label_to_idx": {c: i for i, c in enumerate(shared_pd_lbls)},
        "usage": "Corresponding PlantDoc labels for cross-domain evaluation",
    },
    "zero_shot_pd_labels": {
        "classes": zero_shot_lbls, "num_classes": len(zero_shot_lbls),
        "usage": "PlantDoc-only classes — zero-shot generalisation test",
    },
}
for name, space in eval_spaces.items():
    print(f"  {name:<35} : {space['num_classes']} classes")


  plantvillage_full                   : 37 classes
  cross_domain_pv_labels              : 16 classes
  cross_domain_pd_labels              : 16 classes
  zero_shot_pd_labels                 : 12 classes


## 5. Export Artefacts

In [42]:
# Export mapping CSV (human-readable for supervisor review)
df_mapping = pd.DataFrame(shared)
df_mapping.to_csv(TAXONOMY_OUT / "cross_dataset_mapping.csv", index=False)
print(f"Cross-dataset mapping CSV → outputs/taxonomy/cross_dataset_mapping.csv")

# Export full taxonomy artefacts JSON
artefacts = {
    "metadata": {
        "nomenclature": "EPPO Global Database / USDA PLANTS Database",
        "cross_domain_eval_classes": len(shared),
        "zero_shot_classes": len(pd_only),
    },
    "cross_dataset_mapping": {"shared": shared, "pv_only": pv_only, "pd_only": pd_only},
    "evaluation_label_spaces": eval_spaces,
    "plantdoc_full_class_list": pd_class_names,
}
artefacts_path = TAXONOMY_OUT / "taxonomy_artefacts.json"
with open(artefacts_path, "w") as f:
    json.dump(artefacts, f, indent=2, default=str)
print(f"Taxonomy artefacts → {artefacts_path}")

print("\n✅ SUBTASK 3 COMPLETE")
print(f"   Shared classes        : {len(shared)}")
print(f"   PlantVillage only     : {len(pv_only)}")
print(f"   PlantDoc zero-shot    : {len(pd_only)}")


Cross-dataset mapping CSV → outputs/taxonomy/cross_dataset_mapping.csv
Taxonomy artefacts → D:\IRP\leaf-jepa\stage2_dataset_preparation\outputs\taxonomy\taxonomy_artefacts.json

✅ SUBTASK 3 COMPLETE
   Shared classes        : 16
   PlantVillage only     : 21
   PlantDoc zero-shot    : 12
